In [18]:
# This is the path to the PDF file
pdf_path = "../Book/Arbaeen-Nawawi-book.pdf"


In [19]:
import easyocr

ocr_reader = easyocr.Reader(['ar'])

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [20]:
import fitz
from PIL import Image
import io
import numpy as np

# إصلاح الزخارف التي يقرأها الـ OCR غلط (مثل ﷺ والبسملة المزخرفة)
DECORATION_FIXES = {
    "عطقه": "صلى الله عليه وسلم",
    "عقطته": "صلى الله عليه وسلم",
    "علقاهم": "صلى الله عليه وسلم",
    "عحواله": "صلى الله عليه وسلم",
    "صقاقةهم": "صلى الله عليه وسلم",
    "س إلقوالثفز قليي": "بسم الله الرحمن الرحيم",
}


def fix_decorations(text):
    for wrong, right in DECORATION_FIXES.items():
        text = text.replace(wrong, right)
    return text


def extract_page_lines(pdf_path, page_num, reader, dpi=400):
    """يستخرج سطور صفحة واحدة (page_num يبدأ من 0) بترتيب القراءة العربي."""
    doc = fitz.open(pdf_path)
    pix = doc[page_num].get_pixmap(dpi=dpi)
    image = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
    arr = np.array(image)
    H = arr.shape[0]

    # OCR مع الاحتفاظ بمواضع الكلمات
    results = reader.readtext(arr, detail=1)

    # تجهيز: مركز كل صندوق وارتفاعه وثقته
    items = []
    for bbox, text, conf in results:
        if not text.strip():
            continue
        xs = [p[0] for p in bbox]
        ys = [p[1] for p in bbox]
        items.append({
            "text": text.strip(),
            "cx": sum(xs) / 4,
            "cy": sum(ys) / 4,
            "h": max(ys) - min(ys),
            "conf": float(conf),
        })

    # تجميع الكلمات في سطور (من فوق لتحت)
    items.sort(key=lambda d: d["cy"])
    heights = sorted(d["h"] for d in items)
    med_h = heights[len(heights) // 2]

    def clean_word(w):
        # زخرفة البسملة: صندوق ضخم أعلى الصفحة بثقة منعدمة
        if w["h"] > 2 * med_h and (w["cy"] / H) < 0.25 and w["conf"] < 0.2:
            return "بسم الله الرحمن الرحيم"
        return w["text"]

    lines = []
    for d in items:
        if lines and abs(d["cy"] - lines[-1]["cy"]) < med_h * 0.6:
            lines[-1]["words"].append(d)
            n = len(lines[-1]["words"])
            lines[-1]["cy"] = (lines[-1]["cy"] * (n - 1) + d["cy"]) / n
        else:
            lines.append({"cy": d["cy"], "words": [d]})

    # سطر سطر (وداخل السطر من اليمين لليسار) + إصلاح الزخارف
    out = []
    for line in lines:
        ordered = sorted(line["words"], key=lambda d: -d["cx"])
        text = " ".join(clean_word(w) for w in ordered)
        out.append(fix_decorations(text))
    return out


In [21]:
# مثال: استخراج الصفحة 3
lines_p3 = extract_page_lines(pdf_path, 2, ocr_reader)

print("--- Page 3 OCR (EasyOCR) ---")
for i, line in enumerate(lines_p3, 1):
    print(f"{i:02d} | {line}")


--- Page 3 OCR (EasyOCR) ---
01 | لا عمل إلا بنية
02 | بسم الله الرحمن الرحيم
03 | لا عمل إا بنية
04 | عن أمير المؤمنين آبي حفص عمر بن
05 | الخطاب رضي الله تعالى عنه قال ٠ سمعت رسول
06 | الله صلى الله عليه وسلم يقول ٠ ) إنًمًا الأعمال بالنًيًات وإنًما لكلد
07 | امرئ ما نوًى فمن كانت هجرته إلى اللًه ورسوله
08 | فهجرته إلى اللًه ورسوله ومن كانت هجرته لدنيًا
09 | يصيبها أو امرأة ينكحهًا ؛ فهجرته إل مًا هاجر إليه (
10 | رواه إماما المحدثين أبو عبد اللًه محمد بن إسماعيل
11 | ابن إبراهيم بن المغيرقة بن بردزبه البخاري م وأبو الحسين
12 | مسلم بن الحجًاج بن مسلم القشيريً النًيسابوريً في
13 | صحيحيهما اللذين هما أصح الكتب المصنًفة (١)
14 | (١ أخرجه البخاري في بدء الوحي )ا( ومسلم في الإمارة )١٥٥(
15 | قوله ٠ ل النيات ( أي القصد وعزم القلب على الفعل


In [22]:
for page_num in [2, 3, 4]:
    lines = extract_page_lines(
        pdf_path,
        page_num,
        ocr_reader
    )

    print(f"\n--- Page {page_num + 1} ---")

    for line in lines:
        print(line)


--- Page 3 ---
لا عمل إلا بنية
بسم الله الرحمن الرحيم
لا عمل إا بنية
عن أمير المؤمنين آبي حفص عمر بن
الخطاب رضي الله تعالى عنه قال ٠ سمعت رسول
الله صلى الله عليه وسلم يقول ٠ ) إنًمًا الأعمال بالنًيًات وإنًما لكلد
امرئ ما نوًى فمن كانت هجرته إلى اللًه ورسوله
فهجرته إلى اللًه ورسوله ومن كانت هجرته لدنيًا
يصيبها أو امرأة ينكحهًا ؛ فهجرته إل مًا هاجر إليه (
رواه إماما المحدثين أبو عبد اللًه محمد بن إسماعيل
ابن إبراهيم بن المغيرقة بن بردزبه البخاري م وأبو الحسين
مسلم بن الحجًاج بن مسلم القشيريً النًيسابوريً في
صحيحيهما اللذين هما أصح الكتب المصنًفة (١)
(١ أخرجه البخاري في بدء الوحي )ا( ومسلم في الإمارة )١٥٥(
قوله ٠ ل النيات ( أي القصد وعزم القلب على الفعل

--- Page 4 ---
٤ الآربعين النووية
مراتب الدين
الإسلام  والايمان والاحسان
عن عمر قبه أيضًا قال ٠ بينًما نحن جلوس
عند رسول الله ذاتً يوم إذ طلع علينا رجل شديد
بياض الثًياب شديد سواد الشًعر لا يرى عليه آثر
السًفر ولا يغر فهه منًا أحد حتى جلس إى النبي
فأسند ركبتيه إلى ركبتيه ووضع  كفًيه على فخذيه
وقال ٠ يا محمد ( أخبرني عن الإسلام فقال رسول


In [23]:
import json
import os

import fitz

os.makedirs("../data", exist_ok=True)

doc = fitz.open(pdf_path)
num_pages = len(doc)

all_pages = []
for i in range(num_pages):
    lines = extract_page_lines(pdf_path, i, ocr_reader)
    all_pages.append({"page": i + 1, "lines": lines})
    print(f"Page {i + 1}/{num_pages} done ({len(lines)} lines)")

with open("../data/ocr_pages.json", "w", encoding="utf-8") as f:
    json.dump(all_pages, f, ensure_ascii=False, indent=2)

print("OCR completed successfully.")
print(f"Pages processed: {num_pages}")
print("Saved to: ../data/ocr_pages.json")


Page 1/32 done (8 lines)
Page 2/32 done (21 lines)
Page 3/32 done (15 lines)
Page 4/32 done (17 lines)
Page 5/32 done (17 lines)
Page 6/32 done (17 lines)
Page 7/32 done (16 lines)
Page 8/32 done (16 lines)
Page 9/32 done (16 lines)
Page 10/32 done (16 lines)
Page 11/32 done (16 lines)
Page 12/32 done (17 lines)
Page 13/32 done (15 lines)
Page 14/32 done (16 lines)
Page 15/32 done (17 lines)
Page 16/32 done (16 lines)
Page 17/32 done (16 lines)
Page 18/32 done (18 lines)
Page 19/32 done (16 lines)
Page 20/32 done (17 lines)
Page 21/32 done (17 lines)
Page 22/32 done (17 lines)
Page 23/32 done (17 lines)
Page 24/32 done (16 lines)
Page 25/32 done (16 lines)
Page 26/32 done (16 lines)
Page 27/32 done (17 lines)
Page 28/32 done (17 lines)
Page 29/32 done (16 lines)
Page 30/32 done (16 lines)
Page 31/32 done (17 lines)
Page 32/32 done (26 lines)
OCR completed successfully.
Pages processed: 32
Saved to: ../data/ocr_pages.json
